## 📑Core DPO Low-LTV
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AoAeHg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
%python
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
%python
# Databricks notebook source
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim


import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

##### Schema of the table

In [0]:
%python
table_name = 'Core_DPOLowLTV'

schema = (
"BusinessDate date,"
"ExposureKey string,"
"Assignment string,"
"AssetSegment string,"
"Balance decimal(19,2),"
"Channel string,"
"InteractionTypeKey string,"
"InteractionDate date,"
"ActivityCode string,"
"Category string,"
"Subcategory string,"
"ApplicationsFlag string,"
"ModificationType string,"
"Status string,"
"DateEvaluation string,"
"MonthEvaluation string,"
"ApplicationStep string,"
"CurrentStepDate date,"
"ModsFlag string,"
"ModsMonth string,"
"ModsModificationDate date,"
"ModsProduct string,"
"ModsHaircut decimal(19,2),"
"ModsAmount decimal(19,2),"
"NewExposureKey string,"
"Month string,"
"EbaRisk string,"
"BucketRiskWithMateriality string,"
"CaseBucket string,"
"RiskFound string,"
"OpsFound string,"
"EbaRiskGroup string,"
"BucketRiskGroup string,"
"BucketOperationalGroup string,"
"InsertTimeStamp timestamp"
)

In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
# env_prefix ='qa' #IL: To be Deleted!

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Select tables

In [0]:
%python
assignmentpoollowltvcampaign = spark.sql(
f'''
select
     a.BusinessDate
    ,a.ExposureKey
    ,a.Assignment
    ,a.AssignmentDate
    ,a.Segment
    ,a.AssignmentChannel
    ,a.AssetSegment
    ,a.Balance
from `{env_prefix}-cdh`.externalfiles.extassignmentpoollowltvcampaign a
inner join (select max(IngestionTimeStamp) IngestionTimeStamp, BusinessDate from `{env_prefix}-cdh`.externalfiles.extassignmentpoollowltvcampaign group by BusinessDate) b
on a.BusinessDate = b.BusinessDate and a.IngestionTimeStamp = b.IngestionTimeStamp
where a.BusinessDate = (select max(BusinessDate) from `{env_prefix}-cdh`.externalfiles.extassignmentpoollowltvcampaign where BusinessDate <= {run_date!r})
''')
assignmentpoollowltvcampaign.createOrReplaceTempView("assignmentpoollowltvcampaign")

# Check if the dataset is empty and print a message accordingly
if assignmentpoollowltvcampaign.count() == 0:
    print(f"The required external file extassignmentpoollowltvcampaign dataset for dates <= {run_date!r} is empty")

casecode = spark.sql(
f'''
select distinct d.Portfolioname,  d.exposurekey ,d.casecode
from `{env_prefix}-cdh`.primary.casecode c
inner join `{env_prefix}-cdh`.primary.exposure d on c.casecodekey = d.casecode and c.PortfolioName = d.Portfolioname
where d.Portfolioname = 'Core Retail'
'''
)
casecode.createOrReplaceTempView("casecode")

interaction = spark.sql(
f'''
select
        l.ExposureKey
     , l.PortfolioName
     , l.InteractionDate
     , l.InteractionTypeKey
     , l.InteractionType
     , l.CDI
 FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
 inner join assignmentpoollowltvcampaign a on l.ExposureKey = a.ExposureKey
where 1=1
and InteractionDate <= {run_date!r}
and InteractionDate >= '2022-03-28'
and l.SourceSystem='qca'
and l.PortfolioName = 'Core Retail'
and InteractionTypeKey IN ('152','201','3016','500','501','514'
                                   ,'516','518','521','524','525','528','529','420','531','533','534','536','537','545','546','547','571','572','573','577','6008','6009'
                                   ,'6011','6012','6013','6014','6015','6016','6018','6020','6021','6022','6024','6025','6026'
                                   ,'6027','6028','6029','6030','6035','6036','6037','6038','6039','6040','6041','6042','6044'
                                   ,'6045','6048','6049','6050','6051','6052','6053','6054','6091','6210','6211','6212','6213'
                                   ,'6214','6215','6216','6221','499')
 
union
 
select
       c.ExposureKey
     , l.PortfolioName
     , l.InteractionDate
     , l.InteractionTypeKey
     , l.InteractionType
     , l.CDI
 FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
 inner join casecode c on c.casecode = l.casecode
 inner join assignmentpoollowltvcampaign a on c.ExposureKey = a.ExposureKey
where 1=1
and InteractionDate <= {run_date!r}
and InteractionDate >= '2022-03-28'
and l.SourceSystem='qca'
and l.PortfolioName = 'Core Retail'
and InteractionTypeKey IN ('152','201','3016','500','501','514'
                                   ,'516','518','521','524','525','528','529','420','531','533','534','536','537','545','546','547','571','572','573','577','6008','6009'
                                   ,'6011','6012','6013','6014','6015','6016','6018','6020','6021','6022','6024','6025','6026'
                                   ,'6027','6028','6029','6030','6035','6036','6037','6038','6039','6040','6041','6042','6044'
                                   ,'6045','6048','6049','6050','6051','6052','6053','6054','6091','6210','6211','6212','6213'
                                   ,'6214','6215','6216','6221','499')
''')
interaction.createOrReplaceTempView("interaction")

npeplanningdata = spark.sql(
f'''
select 
               AssetSegment
             , ExposureKey      
             , AgreementKey     
             , Decision			      
             , DateEvaluation       
             , MonthEvaluation    
             , ModificationType     
             , ApplicationNumber    
             , BalanceAmount        
             , AccountBalance
             , Status
             , ApplicationStep     
             , ApplicationStepFinal 
             , ReceiptDateCurrentStep    
 from `{env_prefix}-cdh-delivery`.repoint.core_npeplanningdata
where BusinessDate = {run_date!r}
'''
)
npeplanningdata.createOrReplaceTempView("npeplanningdata")

corestatichistoricalmodificationbeforemigration = spark.sql(
f'''
select 
    RestructuringDate 
    ,OldExposureKey
    ,NewExposureKey
    ,BusinessUnit
    ,RestructuringAmount
    ,HaircutAmount
    ,Category
    ,CategoryFinal
    --,Month
    --,IFRSGrossCarryingAmountPreviousMonthOldExposureKey
from `{env_prefix}-cdh`.reference.corestatichistoricalmodificationbeforemigration
'''
)
corestatichistoricalmodificationbeforemigration.createOrReplaceTempView("corestatichistoricalmodificationbeforemigration")

Core_Modifications = spark.sql(
f'''
select
          ModificationDate
        , BusinessDate
		, OldExposureKey 
		, NewExposureKey 
		, AssetSegment
		, ModificationAmount 
		, HaircutAmount 
		, Category 
		, Categoryfinal 
		, Month
		, IFRSGrossCarryingAmountPrvM
from `{env_prefix}-cdh-delivery`.repoint.core_modifications
where ModificationDate > (select max(RestructuringDate) from corestatichistoricalmodificationbeforemigration) 
AND ModificationDate <= {run_date!r}
'''
)
Core_Modifications.createOrReplaceTempView("Core_Modifications")

dexposure= spark.sql(
f'''
select
     ExposureKey
    ,CDI
    ,Bucket
from `{env_prefix}-cdh`.primary.exposure
where BusinessDate = {run_date!r}
and PortfolioName = 'Core Retail'
and PartyRelationShipType = 'Primary Owner'
and IsActiveFlag = true
'''
)
dexposure.createOrReplaceTempView("dexposure")

unionfile= spark.sql(
f'''
select 
     uf.ExposureNumber
    ,uf.BucketCategoryWithMateriality  
    ,uf.SnapshotDate
    ,uf.EBAStatus
from `{env_prefix}-cdh`.externalfiles.extunionfile uf
join (select SnapshotDate, max(IngestionTimestamp) IngestionTimestamp from `{env_prefix}-cdh`.externalfiles.extunionfile group by SnapshotDate) uf0
on uf.IngestionTimestamp = uf0.IngestionTimestamp and uf.SnapshotDate = uf0.SnapshotDate
where uf.SnapshotDate = (select max(SnapshotDate) from `{env_prefix}-cdh`.externalfiles.extunionfile where SnapshotDate < {run_date!r})

'''
)
unionfile.createOrReplaceTempView("unionfile")

##### Activities

In [0]:
 %python
Activities = spark.sql(
f'''
 SELECT
              a.ExposureKey
            , a.InteractionDate
            , a.InteractionTypeKey
			      , CASE WHEN ac.CDI = a.CDI THEN 1 ELSE 0 END AS PrimaryDebtorFlag
         		, RANK() OVER(PARTITION BY a.ExposureKey ORDER BY a.InteractionDate DESC, CASE WHEN ac.CDI = a.CDI THEN 1 ELSE 0 END DESC, a.InteractionTypeKey DESC) AS Rank
            , am.ActivityCode
            , am.Subcategory
            , am.category
            , am.Channel
    FROM  interaction          a
		LEFT JOIN dexposure ac
		ON ac.ExposureKey = a.ExposureKey --AND ac.CDI = a.CDI
    LEFT JOIN `{env_prefix}-reportinputfiles`.repoint.core_lowltvnactivitymapping am   
    ON a.InteractionTypeKey = am.Code   
    '''
)
Activities.createOrReplaceTempView("Activities")

##### Modifications

In [0]:
 %python
Modifications = spark.sql(
f'''   
 select distinct
               ModificationDate
             , OldExposureKey 
             , NewExposureKey 
             , AssetSegment
             , ModificationAmount 
             , HaircutAmount 
             , Category 
             , Categoryfinal 
from
 (
 SELECT 
		           ModificationDate
             , OldExposureKey
             , NewExposureKey
             , AssetSegment
             , ModificationAmount 
             , HaircutAmount
             , Category
             , Categoryfinal
			       , RANK() OVER(PARTITION BY ModificationDate,OldExposureKey,NewExposureKey  ORDER BY BusinessDate DESC) AS Ranking 
FROM Core_Modifications

UNION ALL

SELECT 
               RestructuringDate
             , OldExposureKey
             , NewExposureKey
             , BusinessUnit AS AssetSegment
             , RestructuringAmount
             , HaircutAmount
             , Category
             , CategoryFinal
             , RANK() OVER(PARTITION BY RestructuringDate,OldExposureKey,NewExposureKey  ORDER BY RestructuringAmount DESC,HaircutAmount DESC) AS Ranking
from corestatichistoricalmodificationbeforemigration
) AS a
WHERE Ranking = 1
    '''
)
Modifications.createOrReplaceTempView("Modifications") 

##### Modifications_V1

In [0]:
  %python
Modifications_V1 = spark.sql(
f'''   
 SELECT DISTINCT
               date_format(m.ModificationDate,'yyyyMM') AS Month
             , m.ModificationDate 
             , m.OldExposureKey
             , m.NewExposureKey
             , m.AssetSegment
             , m.ModificationAmount 
             , m.HaircutAmount
             , m.Category 
             , m.Categoryfinal 
             , RANK() OVER(PARTITION BY m.OldExposureKey ORDER BY m.ModificationDate DESC) AS Rank
        FROM npeplanningdata a
        INNER JOIN Modifications m  
        ON a.ExposureKey = m.OldExposureKey
        WHERE m.ModificationDate > '2022-06-15'
        AND a.ModificationType ILIKE '%discount%'        
    '''
)
Modifications_V1.createOrReplaceTempView("Modifications_V1") 

##### Assignment_V1

In [0]:
%python
Assignment_V1 = spark.sql(
f'''    
SELECT DISTINCT
                a.ExposureKey
              , a.Segment AS Assignment
              , a.AssetSegment
              , a.Balance
              , a.AssignmentChannel    
              , act.InteractionTypeKey
              , act.InteractionDate
              , act.ActivityCode
              , CASE WHEN ap.ModificationType IS NOT NULL THEN 'positive' ELSE act.category END AS category
              , CASE WHEN ap.ModificationType IS NOT NULL THEN 'positive with application' ELSE act.Subcategory END AS Subcategory
              , ap.ModificationType
              , ap.Status
              , ap.DateEvaluation
              , ap.MonthEvaluation
              , ap.ApplicationStep
              , ap.ReceiptDateCurrentStep
              , m.Month               AS ModsMonth
              , m.ModificationDate    AS ModsModificationDate
              , CASE WHEN m.Category = mx.MODs THEN mx.Applications ELSE m.Category END AS ModsProduct
              , m.HaircutAmount       AS ModsHaircut
              , m.ModificationAmount  AS ModsModificationAmount
        FROM assignmentpoollowltvcampaign         a 
        LEFT JOIN Activities act
        ON a.ExposureKey = act.ExposureKey and act.Rank = 1 
        LEFT JOIN npeplanningdata ap
        ON a.ExposureKey = ap.ExposureKey AND ap.ModificationType ILIKE '%discount%' 
        LEFT JOIN Modifications_V1 m
        ON a.ExposureKey = m.OldExposureKey AND m.Rank = 1
        LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.core_lowltvmodsmapping     mx
        ON m.Category = mx.MODs 
    '''
)
Assignment_V1.createOrReplaceTempView("Assignment_V1") 

##### Assignment

In [0]:
%python
Assignment = spark.sql(
f'''      
    SELECT     a1.ExposureKey
             , a1.Assignment
             , a1.AssetSegment
             , replace(replace(a1.Balance,'.',''),',','.') as Balance
             , a1.AssignmentChannel    
             , CASE WHEN category='positive' AND Subcategory='positive with application' AND a1.ModificationType IS NULL                                         THEN NULL
                    WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
             	    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' OR Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' OR Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
               ELSE a1.InteractionTypeKey END AS InteractionTypeKey
             , CASE WHEN category='positive' AND Subcategory='positive with application' AND a1.ModificationType IS NULL                                         THEN NULL
                    WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
             	    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' OR Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' OR Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
               ELSE a1.InteractionDate END AS InteractionDate
             , CASE WHEN category='positive' AND Subcategory='positive with application' AND a1.ModificationType IS NULL                                         THEN NULL
                    WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
             	    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' OR Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' OR Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
               ELSE a1.ActivityCode END AS ActivityCode
             , CASE WHEN category='positive' AND Subcategory='positive with application' AND a1.ModificationType IS NULL                                         THEN NULL
                    WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
             	    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
               ELSE category END AS category
             , CASE WHEN category='positive' AND Subcategory='positive with application' AND a1.ModificationType IS NULL                                         THEN NULL
                    WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
             	    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
               ELSE Subcategory END AS Subcategory
             , CASE WHEN InteractionDate < '2022-08-11' 
             AND 											Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                          THEN NULL 
             	    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
									WHEN DateEvaluation Is Not Null THEN 'Yes'
             	    END AS ApplicationsFlag
			 , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE a1.ModificationType END AS ModificationType
			 , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE a1.Status END AS Status
			 , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE a1.DateEvaluation END AS DateEvaluation
			 , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE a1.MonthEvaluation END AS MonthEvaluation
			  ,	a1.ApplicationStep
              , a1.ReceiptDateCurrentStep
             , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
             	    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
					WHEN ModsModificationDate is not null THEN 'Yes'
             	    END AS ModsFlag
			 , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE ModsMonth END AS ModsMonth
			 , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE ModsModificationDate END AS ModsModificationDate
             , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE ModsProduct END AS ModsProduct
             , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL 
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
             	    WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
             	    WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE ModsHaircut END AS ModsHaircut
             , CASE WHEN InteractionDate < '2022-08-11' AND Assignment='HUBS_LowLTVCampaign_LTV Corrected'                                                        THEN NULL
                    WHEN InteractionDate < '2022-09-27' AND Assignment='HUBS_Low LTV_extra Sep22'                                                                 THEN NULL
        	        WHEN InteractionDate < '2022-10-25' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Oct22_net' Or Assignment='HUBS_LowLTVCampaign_Oct22_net') THEN NULL
        	        WHEN InteractionDate < '2023-04-12' AND (Assignment='HUBS_LowLTVCampaign_Collateral_Apr23_net' Or Assignment='HUBS_LowLTVCampaign_Apr23_net') THEN NULL
                    ELSE ModsModificationAmount END AS ModsModificationAmount
        FROM Assignment_V1   a1
    '''
)
Assignment.createOrReplaceTempView("Assignment") 

##### AssignmentModsPerformance

In [0]:
%python
AssignmentModsPerformance = spark.sql(
f'''    
 SELECT a1.*, m.NewExposureKey, date_FORMAT(m.ModificationDate,'yyyyMM') AS Month
              , CASE WHEN u.ExposureNumber = m.NewExposureKey                   THEN u.EBAStatus
			         WHEN u1.ExposureNumber = a1.ExposureKey AND ModsMonth= date_FORMAT({run_date!r},'yyyyMM') THEN u1.EBAStatus
        	    END AS EbaRisk
             , CASE WHEN u.ExposureNumber = m.NewExposureKey                  THEN u.BucketCategoryWithMateriality 
			        WHEN u1.ExposureNumber = a1.ExposureKey AND ModsMonth= date_FORMAT({run_date!r},'yyyyMM') THEN u1.BucketCategoryWithMateriality	        
        			END AS BucketRiskWithMateriality
             , CASE WHEN a.ExposureKey = m.NewExposureKey THEN a.Bucket END AS CaseBucket
             , CASE WHEN u.ExposureNumber = m.NewExposureKey  THEN 'Yes' 
        	        WHEN u1.ExposureNumber = m.NewExposureKey THEN 'Yes'
        	        END AS RiskFound
             , CASE WHEN a.ExposureKey = m.NewExposureKey THEN 'Yes' END AS OpsFound
        FROM Assignment              a1
        LEFT JOIN Modifications      m  
        ON a1.ExposureKey = m.OldExposureKey AND a1.ModsMonth = date_FORMAT(m.ModificationDate,'yyyyMM')
        AND m.ModificationDate >= to_date(CONCAT(date_format(DATEADD(year, -1, DATE({run_date!r})), 'yyyyMM'), '01'), 'yyyyMMdd') 
        AND m.ModificationDate <= LAST_DAY({run_date!r})
        AND a1.ModsFlag ilike 'yes' AND a1.ApplicationsFlag ilike 'yes'
        LEFT JOIN dexposure            a             
        ON a.ExposureKey = m.NewExposureKey
        LEFT JOIN unionfile        u
        ON u.ExposureNumber = m.NewExposureKey 
        LEFT JOIN unionfile       u1
        ON u1.ExposureNumber = m.NewExposureKey

    '''
)
AssignmentModsPerformance.createOrReplaceTempView("AssignmentModsPerformance") 

##### DPOLowLTV

In [0]:
%python
DPOLowLTV = spark.sql(
f'''        
   SELECT *
             , CASE WHEN EbaRisk='FNPL' OR EbaRisk='NPL' THEN 'NPE'
                    WHEN EbaRisk='PL' THEN 'PL'
        	        WHEN EbaRisk='FPL' THEN 'FPL'
                    END AS EbaRiskGroup
             , CASE WHEN BucketRiskWithMateriality>3 THEN '4+' 
                    WHEN BucketRiskWithMateriality='3' THEN '3'
        	        WHEN BucketRiskWithMateriality='2' THEN '2'
        	        WHEN BucketRiskWithMateriality='1' THEN '1'
        	        WHEN BucketRiskWithMateriality='0' THEN '0'
                    END AS BucketRiskGroup
             , CASE WHEN CaseBucket>'3' OR CaseBucket>'03' THEN '4+'
                    WHEN CaseBucket='3' OR CaseBucket='03' THEN '3'
        	        WHEN CaseBucket='2' OR CaseBucket='02' THEN '2'
        	        WHEN CaseBucket='1' OR CaseBucket='01' THEN '1'
        	        WHEN CaseBucket='0' OR CaseBucket='00' THEN '0'
                    END AS BucketOperationalGroup
        FROM AssignmentModsPerformance 
    '''
)
DPOLowLTV.createOrReplaceTempView("DPOLowLTV") 

##### final_df

In [0]:
%python
final_df = spark.sql(
f'''
SELECT 
    CAST({run_date!r}                  AS DATE)       AS BusinessDate,
    CAST(ExposureKey                   AS STRING)     AS ExposureKey,
    CAST(Assignment                    AS STRING)     AS Assignment,
    CAST(AssetSegment                   AS STRING)     AS AssetSegment,
    CAST(regexp_replace(Balance, '[^0-9].', '') AS DECIMAL(19,2)) AS Balance,
    CAST(AssignmentChannel             AS STRING)     AS Channel,
    CAST(InteractionTypeKey            AS STRING)     AS InteractionTypeKey,
    CAST(InteractionDate               AS DATE)       AS InteractionDate,
    CAST(ActivityCode                  AS STRING)     AS ActivityCode,
    CAST(category                      AS STRING)     AS Category,
    CAST(Subcategory                   AS STRING)     AS Subcategory,
    CAST(ApplicationsFlag              AS STRING)     AS ApplicationsFlag,
    CAST(ModificationType              AS STRING)     AS ModificationType,
    CAST(Status                        AS STRING)     AS Status,
    CAST(DateEvaluation                AS STRING)     AS DateEvaluation,
    CAST(MonthEvaluation               AS STRING)     AS MonthEvaluation,
    CAST(ApplicationStep               AS STRING)     AS ApplicationStep,
    CAST(ReceiptDateCurrentStep        AS DATE)       AS CurrentStepDate,
    CAST(ModsFlag                      AS STRING)     AS ModsFlag,
    CAST(ModsMonth                     AS STRING)     AS ModsMonth,
    CAST(ModsModificationDate          AS DATE)       AS ModsModificationDate,
    CAST(ModsProduct                   AS STRING)     AS ModsProduct,
    CAST(ModsHaircut                   AS DECIMAL(19,2)) AS ModsHaircut,
    CAST(ModsModificationAmount        AS DECIMAL(19,2)) AS ModsAmount,
    CAST(NewExposureKey                AS STRING)     AS NewExposureKey,
    CAST(Month                         AS STRING)     AS Month,
    CAST(EbaRisk                       AS STRING)     AS EbaRisk,
    CAST(BucketRiskWithMateriality     AS STRING)     AS BucketRiskWithMateriality,
    CAST(CaseBucket                    AS STRING)     AS CaseBucket,
    CAST(RiskFound                     AS STRING)     AS RiskFound,
    CAST(OpsFound                      AS STRING)     AS OpsFound,
    CAST(EbaRiskGroup                  AS STRING)     AS EbaRiskGroup,
    CAST(BucketRiskGroup               AS STRING)     AS BucketRiskGroup,
    CAST(BucketOperationalGroup        AS STRING)     AS BucketOperationalGroup,
    CAST(getdate()                     AS TIMESTAMP)  AS InsertTimeStamp
FROM DPOLowLTV A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
%python
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_DPOLowLTV: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = {run_date!r}
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
%python
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
%python
DPOLowLTV = spark.sql(
f'''
SELECT BusinessDate AS ReferenceDate    
     , ExposureKey AS AccountNumber
     , Assignment
     , AssetSegment AS BU
     , Balance/100 as Balance      --Format([Balance], 'N', 'el-GR') [Balance]
     , ModsModificationDate
FROM {delivery_table_path}
WHERE BusinessDate = {run_date!r}
  AND ModsFlag = 'Yes'
GROUP BY  BusinessDate
        , ExposureKey
        , Assignment
        , AssetSegment
        , Balance
        , ModsModificationDate
'''
)
DPOLowLTV.createOrReplaceTempView("DPOLowLTV")

In [0]:
%python
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/DPOLowLTV/"
file_name = "DPOLowLTV_" +run_date+ ".csv"
DPOLowLTV.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)